In [1]:
import sys, os, logging, gc
import numpy as np
from scipy import optimize

from astropy.cosmology import Planck18
import py21cmfast as p21c
from py21cmfast import cache_tools

is_josh = False
if is_josh:
    os.environ['DM21CM_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DM21cm/'
    os.environ['DM21CM_DATA_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/Data01/'
    os.environ['DH_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DarkHistory/'
    os.environ['DH_DATA_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DarkHistory/DHData/'

import numpy as np
from scipy import interpolate
from astropy.cosmology import Planck18
import astropy.units as u

from jax import config
config.update("jax_enable_x64", True)
import jax.numpy as jnp

import py21cmfast as p21c
from py21cmfast import cache_tools

sys.path.append(os.environ['DH_DIR'])
from darkhistory.spec.spectrum import Spectrum

sys.path.append(os.environ['DM21CM_DIR'])
import dm21cm.physics as phys
from dm21cm.dh_wrappers import DarkHistoryWrapper, TransferFunctionWrapper
from dm21cm.utils import load_h5_dict
from dm21cm.data_cacher import Cacher
from dm21cm.profiler import Profiler

logging.getLogger().setLevel(logging.INFO)
logging.getLogger('21cmFAST').setLevel(logging.CRITICAL+1)
logging.getLogger('py21cmfast._utils').setLevel(logging.CRITICAL+1)
logging.getLogger('py21cmfast.wrapper').setLevel(logging.CRITICAL+1)

#######################################
###   Import and Construct DM21cm   ###
#######################################

cache_dir = os.environ['P21C_CACHE_DIR']

os.environ['P21C_CACHE_DIR'] = cache_dir
p21c.config['direc'] = os.environ['P21C_CACHE_DIR']
WDIR = os.environ['DM21CM_DIR']
sys.path.append(WDIR)
from dm21cm.dm_params import DMParams
#from dm21cm.evolve import evolve

use_tqdm = True

/n/home07/yitians/.conda/envs/dm21cm/lib/python3.11/site-packages/py21cmfast/_cfg.py:58: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn(
/n/home07/yitians/.conda/envs/dm21cm/lib/python3.11/site-packages/py21cmfast/_cfg.py:42: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn("Your configuration file is out of date. Updating...")


In [2]:
from dm21cm.evolve import get_z_edges, split_xray, gen_injection_boxes

In [3]:
run_name = 'test'
z_start = 45.
z_end = 40.
dm_params = DMParams(
    mode='decay',
    primary='phot_delta',
    m_DM=1e8, # [eV]
    lifetime=1e31, # [s]
)
enable_elec = False

p21c_initial_conditions = p21c.initial_conditions(
    user_params = p21c.UserParams(
        HII_DIM = 32,
        BOX_LEN = 32*2, # [conformal Mpc]
        N_THREADS = 32,
    ),
    cosmo_params = p21c.CosmoParams(
        OMm = Planck18.Om0,
        OMb = Planck18.Ob0,
        POWER_INDEX = Planck18.meta['n'],
        SIGMA_8 = Planck18.meta['sigma8'],
        hlittle = Planck18.h,
    ),
    random_seed = 54321,
    write = True,
)
# p21c_astro_params = p21c.AstroParams._defaults_
# astro_params = p21c_astro_params
astro_params = None
p21c_astro_params = None
clear_cache = True
use_DH_init = False
no_injection = False
tf_on_device = True
rerun_DH = False
use_xray_interp_shell = True

/n/home07/yitians/.conda/envs/dm21cm/lib/python3.11/site-packages/py21cmfast/inputs.py:487: UserWarning: The USE_INTERPOLATION_TABLES setting has changed in v3.1.2 to be default True. You can likely ignore this warning, but if you relied onhaving USE_INTERPOLATION_TABLES=False by *default*, please set it explicitly. To silence this warning, set it explicitly to True. Thiswarning will be removed in v4.
  warnings.warn(


In [4]:
def p21c_step(perturbed_field, spin_temp, ionized_box,
             input_heating=None, input_ionization=None, input_jalpha=None, astro_params=astro_params):

    spin_temp = p21c.spin_temperature(
        perturbed_field = perturbed_field,
        previous_spin_temp = spin_temp,
        input_heating_box = input_heating,
        input_ionization_box = input_ionization,
        input_jalpha_box = input_jalpha,
        astro_params = astro_params,
    )

    ionized_box = p21c.ionize_box(
        perturbed_field = perturbed_field,
        previous_ionize_box = ionized_box,
        spin_temp = spin_temp,
        astro_params = astro_params,
    )

    brightness_temp = p21c.brightness_temperature(
        ionized_box = ionized_box,
        perturbed_field = perturbed_field,
        spin_temp = spin_temp,
    )

    return spin_temp, ionized_box, brightness_temp

In [5]:
#===== data and cache =====
p21c.config[f'direc'] = cache_dir = '/n/home07/yitians/21cmFAST-cache/test_new_evolve'
data_dir = os.environ['DM21CM_DATA_DIR'] + '/tf/zf01/data'
logging.info(f"Cache dir: {cache_dir}")
os.makedirs(cache_dir, exist_ok=True)
if clear_cache:
    cache_tools.clear_cache()
gc.collect()


#===== initialize =====
#--- physics parameters ---
abscs = load_h5_dict(f"{data_dir}/abscissas.h5")
dm_params.set_inj_specs(abscs)

EPSILON = 1e-6
p21c.global_params.Z_HEAT_MAX = z_start + EPSILON
p21c.global_params.ZPRIME_STEP_FACTOR = abscs['zplusone_step_factor']

box_dim = p21c_initial_conditions.user_params.HII_DIM
box_len = p21c_initial_conditions.user_params.BOX_LEN
cosmo = Planck18

#--- DarkHistory and transfer functions ---
tf_wrapper = TransferFunctionWrapper(
    box_dim = box_dim,
    abscs = abscs,
    prefix = data_dir,
    enable_elec = enable_elec,
    on_device = tf_on_device,
)

#--- xray ---
if use_xray_interp_shell:
    xray_cacher = Cacher(box_dim=box_dim, dx=box_len/box_dim)
else:
    from dm21cm.data_cacher_old import Cacher
    xray_cacher = Cacher(data_path=f"{cache_dir}/xray_brightness.h5", cosmo=cosmo, N=box_dim, dx=box_len/box_dim)
    xray_cacher.clear_cache()

#--- redshift stepping ---
z_edges = get_z_edges(z_start, z_end, p21c.global_params.ZPRIME_STEP_FACTOR)

#===== initial steps =====
dh_wrapper = DarkHistoryWrapper(dm_params, prefix=p21c.config[f'direc'])

# We have to synchronize at the second step because 21cmFAST acts weird in the first step:
# - global_params.TK_at_Z_HEAT_MAX is not set correctly (it is probably set and evolved for a step)
# - global_params.XION_at_Z_HEAT_MAX is not set correctly (it is probably set and evolved for a step)
# - first step ignores any values added to spin_temp.Tk_box and spin_temp.x_e_box
z_match = z_edges[1]
if use_DH_init:
    dh_wrapper.evolve(end_rs=(1+z_match)*0.9, rerun=rerun_DH)
    T_k_DH_init, x_e_DH_init, phot_bath_spec = dh_wrapper.get_init_cond(rs=1+z_match)
else:
    phot_bath_spec = Spectrum(abscs['photE'], np.zeros_like(abscs['photE']), spec_type='N', rs=1+z_match) # [ph / Bavg]

perturbed_field = p21c.perturb_field(redshift=z_edges[1], init_boxes=p21c_initial_conditions, write = True)
spin_temp, ionized_box, brightness_temp = p21c_step(perturbed_field=perturbed_field, spin_temp=None, ionized_box=None, astro_params=p21c_astro_params)
if use_DH_init:
    spin_temp.Tk_box += T_k_DH_init - np.mean(spin_temp.Tk_box)
    spin_temp.x_e_box += x_e_DH_init - np.mean(spin_temp.x_e_box)
    ionized_box.xH_box = 1 - spin_temp.x_e_box

INFO:root:Cache dir: /n/home07/yitians/21cmFAST-cache/test_new_evolve


INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': NOT_FOUND: Could not find registered platform with name: "rocm". Available platform names are: CUDA Interpreter
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': module 'jaxlib.xla_extension' has no attribute 'get_tpu_client'
INFO:root:TransferFunctionWrapper: Loaded photon transfer functions.
INFO:root:TransferFunctionWrapper: Skipping electron transfer functions.


zp = 4.474240e+01 E_tot_ave = 0.000000e+00


In [6]:
def get_r_shells(box_dim, box_len, r_cap=None, n_target=40):
    """Generate r values for interpolation shells."""
    L_FACTOR = 0.620350491
    R = L_FACTOR * box_len/box_dim
    R_factor = (p21c.global_params.R_XLy_MAX/R) ** (1/p21c.global_params.NUM_FILTER_STEPS_FOR_Ts)
    r_s = R * R_factor**np.arange(n_target)
    r_s = np.append(r_s, p21c.global_params.R_XLy_MAX)
    r_s = np.insert(r_s, 0, 0.)
    if r_cap is not None:
        r_max = np.min([r_cap, box_len/2])
    else:
        r_max = box_len/2
    r_s = np.unique(np.minimum(r_s, r_max)) # smooth up to radii at half the box length
    return r_s

In [7]:
#===== main loop =====
z_edges = z_edges[1:] # Maybe fix this later
z_range = range(len(z_edges)-1)
records = []
profiler = Profiler()

#--- trackers ---
i_xray_loop_start = 0 # where we start looking for annuli

In [8]:
i_z = 0

In [12]:
i_z += 1

In [13]:
print(f'i_z={i_z}/{len(z_edges)-2} z={z_edges[i_z]:.2f}')

#===== physical quantities =====
z_current = z_edges[i_z]
z_next = z_edges[i_z+1]
dt = phys.dt_step(z_current, np.exp(abscs['dlnz']))

#--- for interpolation ---
delta_plus_one_box = 1 + np.asarray(perturbed_field.density)
x_e_box = np.asarray(1 - ionized_box.xH_box)
T_k_box = np.asarray(spin_temp.Tk_box)
tf_wrapper.init_step(rs=1+z_current, delta_plus_one_box=delta_plus_one_box, x_e_box=x_e_box, T_k_box=T_k_box)

#--- for dark matter ---
nBavg = phys.n_B * (1+z_current)**3 # [Bavg / (physical cm)^3]
rho_DM_box = delta_plus_one_box * phys.rho_DM * (1+z_current)**3 # [eV/(physical cm)^3]
inj_per_Bavg_box = phys.inj_rate(rho_DM_box, dm_params) * dt * dm_params.struct_boost(1+z_current) / nBavg # [inj/Bavg]

i_z=1/10 z=44.29


In [14]:
len(xray_cacher.states)==0

False

In [22]:
r_from_z = np.vectorize(lambda z: phys.conformal_dx_between_z(z_current, z)) # conformal distance [cMpc] of z from current shell
z_interp_arr = np.geomspace(z_current, 200., 1000) # up to matter CMB decoupling
r_interp_arr = r_from_z(z_interp_arr)
z_from_r = interpolate.interp1d(r_interp_arr, z_interp_arr, bounds_error=False, fill_value='extrapolate') # inverse of r_z

r_shells = get_r_shells(box_dim, box_len, r_cap=r_from_z(np.max(xray_cacher.z_s)), n_target=40) # R_a in paper
z_shells = z_from_r(r_shells) # z_a in paper
z_shell_mids = np.concatenate([[z_current], (z_shells[:-1] + z_shells[1:]) / 2, [z_shells[-1]]])
r_shell_mids = r_from_z(z_shell_mids) # start and end for R windows
dz_shells = np.diff(z_shell_mids) # dz_a in paper

In [23]:
xray_cacher.z_s

array([44.74240221])

In [27]:
dz_shells

array([0.02386622, 0.02772786, 0.00834809, 0.00970769, 0.01129044,
       0.01312045, 0.01524338, 0.01771276, 0.02060283, 0.02395757,
       0.02784888, 0.03239113, 0.03767737, 0.04382366, 0.05099946,
       0.05801361, 0.03057803])

In [28]:


# Example (make a better comment later)
# inds         =   0, 1,   2,   ..., N-2, N-1    |
# r_shells     =   0, 1.2, 2.4, ..., 250, 256    | total=N
# r_shell_mids = 0, 0.6, 1.8, ..., 247, 253, 256 | total=N+1

for i, z_shell in enumerate(z_shells):

    # z_left < z_shell < z_right
    i_z_right = np.searchsorted(-z_edges, -z_shell, side='right')
    i_z_left = i_z_right + 1
    z_right = z_edges[i_z_right]
    z_left = z_edges[i_z_left]

    ftdEdz_right, rel_spec_right = xray_cacher.get_ftdEdz_spec(z_right)
    ftdEdz_left,  rel_spec_left  = xray_cacher.get_ftdEdz_spec(z_left)
    left_weight = (z_right - z_shell) / (z_right - z_left)
    right_weight = 1 - left_weight

    ftdEdz = left_weight * ftdEdz_left + right_weight * ftdEdz_right
    dEdz, _ = xray_cacher.smooth_box(ftdEdz, r_shell_mids[i], r_shell_mids[i+1]) # r_shell_mids[i] < r_shell < r_shell_mids[i+1]
    rel_spec = left_weight * rel_spec_left + right_weight * rel_spec_right

    dE = dEdz * dz_shells[i] # [eV/Bavg]
    tf_wrapper.inject_phot(rel_spec, inject_type='xray', weight_box=dE)

# We have summed all the shells up to r_shells[-1] (precisely), and we need to release the rest to bath
phot_bath_spec += xray_cacher.release_to_bath_prior_to(z_shells[-1])

ValueError: z=43.8410961800588 out of bounds 44.742402213277984 - 44.742402213277984.

In [20]:
dz_shells

array([0.])

In [ ]:

        ################################################
        ###   Begin New X-Ray Deposition Treatment   ###
        ################################################
        
        print('On step:', i_z, 'have', len(xray_cacher.brightness_cache.z_s), 'cached states')
        if len(xray_cacher.brightness_cache.z_s) > 0:

            # These are the redshifts of emission corresponding to the radii we intend to inspect.
            # We evalute both the emission edges associated with the annulus radii and the midpoint
            z_annulus_edges = np.array([get_z_emission(z_current, r) for r in r_vals])
            z_annulus_mid = np.array([get_z_emission(z_current, r) for r in r_mid])

            # This sets up the lookback pairs based on the cached data.
            z_lookback_pairs = make_z_pairs(xray_cacher.brightness_cache.z_s, z_annulus_edges, z_annulus_mid)
    
            for i in range(len(z_lookback_pairs)):
                print('Smooth on:', r_pairs[i])
                print('Interpolation Endpoints:', z_lookback_pairs[i][0], z_lookback_pairs[i][1])
                print('Interpolation Midpoint:', z_annulus_mid[i])
                print()
            
            for i in range(len(z_lookback_pairs)):
                # Get the left deposition
                left_box, left_spec = new_smoothed_annulus(self, z_lookback_pairs[i][0],
                                                           r_pairs[i][0], r_pairs[i][1])
                right_box, right_spec = new_smoothed_annulus(self, z_lookback_pairs[i][1],
                                                             r_pairs[i][0], r_pairs[i][1])
                
                left_weight = np.interp(z_mid[i], z_lookback_pairs[i], [1, 0])
                right_weight = np.interp(z_mid[i], z_lookback_pairs[i], [0, 1])
                
                tf_wrapper.inject_phot(left_spec, inject_type='xray', weight_box=left_weight*left_box)
                tf_wrapper.inject_phot(right_spec, inject_type='xray', weight_box=right_weight*right_box)
                
        profiler.record('xray')
    
        ##############################################
        ###   End New X-Ray Deposition Treatment   ###
        ##############################################
        
        ##################################
        ###   Need a bath check here   ###
        ##################################

        # Boolean array inside xray cacher indicating whether or not the state was accessed in this step.
        # Boolean array inside xray cacher indicating if the spectrum has already been dumped to bath.
        
        # If not accessed and not in bath. Add to bath. MAKE SURE TO ADD TO BATH BEFORE BATH INJECTION.
        # Inside xray cacher, exclude the spectra that have been dumped to bath from the advance_spectrum method.
        
        ##############################
        ###   END THE BATH CHECK   ###
        ##############################


        

In [11]:
#--- bath and homogeneous portion of xray ---
if not no_injection:
    tf_wrapper.inject_phot(phot_bath_spec, inject_type='bath')

    #--- dark matter (on-the-spot) ---
    tf_wrapper.inject_from_dm(dm_params, inj_per_Bavg_box)

#===== 21cmFAST step =====
perturbed_field = p21c.perturb_field(redshift=z_next, init_boxes=p21c_initial_conditions)
input_heating, input_ionization, input_jalpha = gen_injection_boxes(z_next, p21c_initial_conditions)
tf_wrapper.populate_injection_boxes(input_heating, input_ionization, input_jalpha, dt,)
spin_temp, ionized_box, brightness_temp = p21c_step(
    perturbed_field, spin_temp, ionized_box,
    input_heating = input_heating,
    input_ionization = input_ionization,
    input_jalpha = input_jalpha,
    astro_params = p21c_astro_params
)

#===== prepare spectra for next step =====
#--- bath (separating out xray) ---
prop_phot_N = np.array(tf_wrapper.prop_phot_N) # propagating and emitted photons have been stored in tf_wrapper up to this point, time to get them out
emit_phot_N = np.array(tf_wrapper.emit_phot_N)
emit_bath_N, emit_xray_N = split_xray(emit_phot_N, abscs['photE'])
phot_bath_spec = Spectrum(abscs['photE'], prop_phot_N + emit_bath_N, rs=1+z_current, spec_type='N') # photons not emitted to the xray band are added to the bath (treated as uniform)
phot_bath_spec.redshift(1+z_next)

#--- xray ---
x_e_for_attenuation = 1 - np.mean(ionized_box.xH_box)
attenuation_arr = np.array(tf_wrapper.attenuation_arr(rs=1+z_current, x=np.mean(x_e_for_attenuation))) # convert from jax array
xray_cacher.advance_spectra(attenuation_arr, z_next)

xray_spec = Spectrum(abscs['photE'], emit_xray_N, rs=1+z_current, spec_type='N') # [ph/Bavg]
xray_spec.redshift(1+z_next)
xray_tot_eng = np.dot(abscs['photE'], emit_xray_N)
if xray_tot_eng == 0.:
    xray_rel_eng_box = np.zeros_like(tf_wrapper.xray_eng_box)
else:
    xray_rel_eng_box = tf_wrapper.xray_eng_box / xray_tot_eng # [1 (relative energy)/Bavg]
if not no_injection:
    #I CHANGED THIS. THIS USED TO BE Z_CURRENT BUT I THINK THIS IS WRONG!!!!!!!!!
    xray_cacher.cache(z_next, z_current, xray_spec, xray_rel_eng_box)

#===== calculate and save some quantities =====
dE_inj_per_Bavg = dm_params.eng_per_inj * np.mean(inj_per_Bavg_box) # [eV/Bavg]
dE_inj_per_Bavg_unclustered = dE_inj_per_Bavg / dm_params.struct_boost(1+z_current) # [eV/Bavg]

records.append({
    'z'   : z_next,
    'T_s' : np.mean(spin_temp.Ts_box), # [mK]
    'T_b' : np.mean(brightness_temp.brightness_temp), # [K]
    'T_k' : np.mean(spin_temp.Tk_box), # [K]
    'x_e' : np.mean(spin_temp.x_e_box), # [1]
    '1-x_H' : np.mean(1 - ionized_box.xH_box), # [1]
    'E_phot' : phot_bath_spec.toteng(), # [eV/Bavg]
    'phot_N' : phot_bath_spec.N, # [ph/Bavg]
    'dE_inj_per_B' : dE_inj_per_Bavg, # [eV/Bavg]
    'dE_inj_per_Bavg_unclustered' : dE_inj_per_Bavg_unclustered, # [eV/Bavg]
    'dep_ion'  : np.mean(tf_wrapper.dep_box[...,0] + tf_wrapper.dep_box[...,1]), # [eV/Bavg]
    'dep_exc'  : np.mean(tf_wrapper.dep_box[...,2]), # [eV/Bavg]
    'dep_heat' : np.mean(tf_wrapper.dep_box[...,3]), # [eV/Bavg]
    'x_e_slice' : np.array(spin_temp.x_e_box[0]), # [1]
    'x_H_slice' : np.array(ionized_box.xH_box[0]), # [1]
    'T_k_slice' : np.array(spin_temp.Tk_box[0]), # [K]
})

zp = 4.428951e+01 E_tot_ave = 0.000000e+00


In [ ]:
def get_r_emission(z_receiver, z_emission):
    '''
    Calculates the conformal distance that photons emitted at z_emission travel over the
    interval z_receiver -> z_emission
    '''
    
    return np.abs(phys.conformal_dt_between_z(z_receiver, z_emission)) * phys.c / phys.Mpc

def get_z_emission(z_receiver, r):
    '''
    Calculates the redshift of emission for photons that travel a comoving distance `r`
    before they arrive to deposit at time `z_receiver`
    '''
    
    return optimize.brentq(lambda z_emission: get_r_emission(z_receiver, z_emission) - r, z_receiver, 1000)

def make_z_pairs(z_prior, z_annulus_edges, z_annulus_mid):
    z_lookback_pairs = []
    
    for i in range(len(z_annulus_mid)):

        # This is the left endpoint for our interpolation. z_left
        # is closer to the z_deposition than the midpoint of our annulus
        z_left = np.amax(z_prior[np.where(z_prior < z_annulus_mid[i])])

        # This is the right endpoint for our interpolation. z_right'
        # is further from z_deposition than the mdpoint of our annulus.
        # We handle it carefully in case there is nothing in the array that
        # satisfies this.
        z_right = z_prior[np.where(z_prior > z_annulus_mid[i])]

        if len(z_right)> 0:
            z_right = np.amin(z_right)
        else:
            z_right = z_annulus_edges[i+1]

        z_lookback_pairs.append([z_left, z_right])
        
    return z_lookback_pairs


global_params = p21c.global_params

L_FACTOR = 0.620350491
HII_DIM = spin_temp.user_params.HII_DIM
BOX_LEN = spin_temp.user_params.BOX_LEN
R = L_FACTOR* BOX_LEN/HII_DIM
R_factor = pow(global_params.R_XLy_MAX/R, 1/global_params.NUM_FILTER_STEPS_FOR_Ts)

# Get all the 21cmFAST smoothing radii
r_vals = R*R_factor**np.arange(40)
r_vals = np.append(r_vals, global_params.R_XLy_MAX)

# Only smooth up to radii at half the box length
r_vals = np.unique(np.minimum(np.append(0, r_vals), BOX_LEN/2)) # These are the annuli we will smooth on

r_pairs = []

for i in range(len(r_vals)-1):
    r_pairs.append([r_vals[i], r_vals[i+1]])
    
r_mid = (r_vals[1:] + r_vals[:-1])/2